# Household MPC Experiment

Dedicated paired baseline-versus-income-shock workflow. The implementation lives in `src/experiments/mpc.py`; the historical command remains available through `run_mpc_experiment.py`.

USES THE BASELINE CONFIG, no overrides

In [1]:
# Setup
%load_ext autoreload
%autoreload 2

import pandas as pd
from src.experiments.mpc import run_mpc_experiment
from src.mpc_analysis import MPCFilterConfig, plot_mpc_panel

## Inputs

In [2]:
RUN_MPC = True
FORCE_REBUILD_DATA = True  # Rebuild prepared data.pkl; baseline and shock simulations run anew.
COUNTRY = "FRA"
SEEDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
T_MAX = 80
SHOCK_PERIOD = 60
HORIZON_PERIODS = 4
SHOCK_FRACTION = 0.1
N_JOBS = 4
OUTPUT_DIR = "data/output_data/mpc-experiment"

filters = MPCFilterConfig(
    min_income=1000,
    income_top_quantile=0.99,
    consumption_income_ratio_quantiles=(0.01, 0.99),
    gross_wealth_top_quantile=0.999,
    head_age_min=25,
    head_age_max=75,
)
# Explicit current outputs; remove names here to save fewer series.
OUTPUT_VARIABLES = [
    "cash",
    "total",
    "behavioural_total_target",
    "imputed_rent",
    "target_consumption_total_mpc",
]

## Run

In [ ]:
outputs = None
if RUN_MPC:
    outputs = run_mpc_experiment(
        paper_outputs=True,
        output_variables=OUTPUT_VARIABLES,
        seeds=SEEDS,
        t_max=T_MAX,
        shock_period=SHOCK_PERIOD,
        horizon_periods=HORIZON_PERIODS,
        shock_fraction=SHOCK_FRACTION,
        country_iso3=COUNTRY,
        output_dir=OUTPUT_DIR,
        n_jobs=N_JOBS,
        force_rebuild_data=FORCE_REBUILD_DATA,
        apply_mpc_filters=True,
        mpc_filter_config=filters,
        stay_in_activity_bracket_only=False,
        stable_effective_labor_state_only=False,
        # Impact MPC on every household. The cumulative MPC is identified where the
        # labour-income gap between the two runs is at most 5% of the shock.
        labor_income_tolerance=0.05,
        # False: support counts the transfer as a resource. True: the support test ignores it.
        shock_exempt_from_support=True,
    )
else:
    print("Set RUN_MPC = True to run the experiment.")

{'seed': 1, 'country': 'FRA', 't_max': 80, 'raw_data_path': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/raw_data', 'output_dir': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/mpc-experiment', 'data_cache': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/mpc-experiment/data.pkl'}
Configuration summary
{'productivity_growth': 'SimpleTFPGrowth',
 'productivity_investment_planner': 'TargetIntensityTFPInvestmentPlanner',
 'labour_market': {'name': 'ReservationWageBindingDefaultLabourMarketClearer',
                   'parameters': {'allow_switching_industries': True,
                                  'compare_with_normalised_inputs': True,
                                  'consider_reservation_wages': True,
                                  'firing_cost_fraction': 0.0,
                                  'firing_speed': 1.0,
                                  'hiring_cost_fraction': 0.0,
 

/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/firms/func/prices.py:734: RuntimeWarning: Markup residual calibration: 2 sector(s) have an AC floor above their initial-price anchor and cannot reach it at any residual factor: industry 1 (+3.36%), industry 11 (+5.31%). These sectors keep their unadjusted Orbis markup and price at average cost until unit depreciation falls relative to marginal cost. The percentage is the AC-floor overshoot of the anchor, also recorded as pricing_markup_residual_unreachable_gap.
  self._warn_on_unreachable_markup_anchors()
/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/firms/func/prices.py:734: RuntimeWarning: Markup residual calibration: 2 sector(s) have an AC floor above their initial-price anchor and cannot reach it at any residual factor: industry 1 (+3.36%), industry 11 (+5.31%). These sectors keep their unadjusted Orbis markup and price at average cost until unit depreciation falls relative to ma

## Results and plots

Income panels use household income components recorded before the transfer: total income, employee (labour) income, non-property income (employee plus social transfers), and property income (rental plus financial-asset income or dividends, following the model setting). The plotted sample is unweighted. Zero and negative component values are shown as separate groups; quintiles are formed among positive values.


In [ ]:
if outputs is not None:
    panel = pd.read_csv(outputs["analysis_dir"] / "household_mpc_panel.csv")
    summary = pd.read_csv(outputs["analysis_dir"] / "household_mpc_summary.csv")
    identified_cumulative = panel["cumulative_mpc_identified"].fillna(False).astype(bool)
    # identified_cumulative = pd.Series(True, index=panel.index)
    plot_panel = panel.copy()
    plot_panel["real_cmpc_4q"] = plot_panel["real_cmpc_4q"].where(identified_cumulative)
    plot_mpc_panel(
        plot_panel,
        variables=[
            "income",
            "net_wealth",
            "housing_tenure",
            "net_liquid_financial_assets",
            "illiquid_financial_assets",
            "housing_wealth",
        ],
        mpc_columns=["real_mpc_impact", "real_cmpc_4q"],
        # mpc_columns=["cash_real_mpc_impact", "cash_real_cmpc_4q"],
        plot_kind="box_only",
        y_range=(-0.5, 1.1),
        winsor_quantiles=(0, 1),
        # winsor_quantiles=(0.01, 0.99),
        winsor_mode="trim",
        output_dir=outputs["analysis_dir"] / "plots",
        output_format="png",
    )
    summary

### Slide plot: impact and 4-quarter MPC by income and liquidity quintile

In [ ]:
# Paper figure: real impact MPCs after a temporary, equal-per-household income transfer.
# Boxes use the filtered household-seed panel; baseline states classify each household before the transfer.
if outputs is not None:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from src.mpc_analysis import assign_quantile_bin

    panel = pd.read_csv(outputs["analysis_dir"] / "household_mpc_panel.csv")
    def income_group(values):
        """Keep zero/negative masses intact; quintile-rank positive incomes."""
        values = pd.to_numeric(values, errors="coerce")
        groups = pd.Series(pd.NA, index=values.index, dtype="string")
        groups.loc[values < 0] = "Negative"
        groups.loc[values == 0] = "Zero"
        positive = values > 0
        groups.loc[positive] = assign_quantile_bin(values.loc[positive]).astype("string")
        return groups

    panel["total_income_bin"] = income_group(panel["income"])
    panel["labor_income_bin"] = income_group(panel["labor_income"])
    panel["non_property_income_bin"] = income_group(panel["non_property_income"])
    panel["property_income_bin"] = income_group(panel["property_income"])
    panel["lfa_bin"] = assign_quantile_bin(panel["liquid_financial_assets"])
    panel["nla_all_debt"] = panel["liquid_financial_assets"] - panel["debt"]
    panel["nla_all_debt_bin"] = assign_quantile_bin(panel["nla_all_debt"])
    panel["ifa_positive_bin"] = pd.NA
    ifa_positive = panel["illiquid_financial_assets"] > 0
    panel.loc[ifa_positive, "ifa_positive_bin"] = assign_quantile_bin(panel.loc[ifa_positive, "illiquid_financial_assets"])
    panel["ha_positive_bin"] = pd.NA
    ha_positive = panel["housing_wealth"] > 0
    panel.loc[ha_positive, "ha_positive_bin"] = assign_quantile_bin(panel.loc[ha_positive, "housing_wealth"])

    activity_labels = {1: "Employed", 2: "Unemployed", 3: "Inactive", 4: "Retired"}
    panel["labour_status"] = panel["activity_bracket_t0"].map(activity_labels).fillna("Other / missing")

    tenure_labels = {
        "renter": "Renter",
        "owner_mortgage": "Mortgagors",
        "owner_outright": "Outright owners",
        "owner_other": "Other",
        "unknown": "Other",
    }
    panel["tenure_label"] = panel["housing_tenure"].map(tenure_labels).fillna("Other / missing")

    # Debt-type classification uses opening consumer and mortgage balances.
    consumer_debt = panel["consumption_loan_debt"] > 1e-8
    mortgage_debt = panel["mortgage_debt"] > 1e-8
    panel["borrowing_type"] = "No borrowing"
    panel.loc[consumer_debt & ~mortgage_debt, "borrowing_type"] = "Consumer credit"
    panel.loc[~consumer_debt & mortgage_debt, "borrowing_type"] = "Mortgage"
    panel.loc[consumer_debt & mortgage_debt, "borrowing_type"] = "Both"

    # Calibration profile rules from the weighted FRA household-group audit.
    # Assets in this simulation are scaled by 5,000. The MPC panel only saves
    # current quarterly income, used here as an annual-income proxy (4 quarters).
    # Calibration HA is main-residence wealth; housing_wealth here also includes
    # other properties, so the wealthy-type split is a documented proxy.
    asset_scale = 5000.0
    lfa_cal = panel["liquid_financial_assets"] / asset_scale
    ifa_cal = panel["illiquid_financial_assets"] / asset_scale
    ha_cal_proxy = panel["housing_wealth"] / asset_scale
    annual_income_proxy = panel["income"] * 4.0 / asset_scale
    positive_income = annual_income_proxy > 0
    poor_htm = (lfa_cal <= 1000.0) & (ifa_cal <= 0.0) & (ha_cal_proxy <= 0.0)
    wealthy_htm = (
        positive_income
        & (lfa_cal <= (3.0 / 12.0) * annual_income_proxy)
        & ((ifa_cal + ha_cal_proxy) >= 2.0 * annual_income_proxy)
    )
    doubly_wealthy = (lfa_cal > 11153.0) & (ifa_cal > 1638.0) & (ha_cal_proxy > 141865.0)
    # Make the display groups exclusive; give the more specific double-wealth
    # definition precedence where the calibration masks overlap.
    panel["household_type"] = "Other"
    panel.loc[wealthy_htm, "household_type"] = "Rich HtM"
    panel.loc[doubly_wealthy, "household_type"] = "Doubly wealthy"
    panel.loc[poor_htm, "household_type"] = "Poor HtM"

    # Reconstruct baseline goods-demand cuts directly from each saved baseline window.
    panel["baseline_goods_cut"] = np.nan
    for seed, rows in panel.groupby("seed", sort=False):
        window_path = outputs["baseline_dir"] / f"seed-{int(seed)}" / "mpc_window.npz"
        with np.load(window_path) as window:
            cut = np.asarray(window["regime_consumption_cut_amount"], dtype=float)
            floor = np.asarray(window["regime_subsistence_floor"], dtype=float)
        ids = rows["household_id"].astype(int).to_numpy()
        panel.loc[rows.index, "baseline_goods_cut"] = cut[ids]
        panel.loc[rows.index, "baseline_goods_cut_tolerance"] = 1e-9 * np.maximum(np.abs(floor[ids]), 1.0)
    panel["liquidity_status"] = pd.NA
    ordinary_regimes = panel["mpc_regime"].isin(["liquidity_constrained", "unconstrained"])
    cut_binds = panel["baseline_goods_cut"] > panel["baseline_goods_cut_tolerance"]
    panel.loc[ordinary_regimes & cut_binds, "liquidity_status"] = "Liquidity constrained"
    panel.loc[panel["mpc_regime"].eq("unconstrained"), "liquidity_status"] = "Not constrained"

    quintiles = ["Q1", "Q2", "Q3", "Q4", "Q5"]
    income_groups = ["Negative", "Zero", *quintiles]
    plot_specs = [
        ("total_income_bin", "Total income", income_groups, None),
        ("labor_income_bin", "Labour income", income_groups, None),
        ("non_property_income_bin", "Non-property income", income_groups, None),
        ("property_income_bin", "Property income", income_groups, None),
        ("lfa_bin", "Liquid financial assets", quintiles, None),
        ("nla_all_debt_bin", "Net liquid assets", quintiles, None),
        ("ifa_positive_bin", "Illiquid financial assets", quintiles, ifa_positive),
        ("ha_positive_bin", "Housing wealth", quintiles, ha_positive),
        ("tenure_label", "Housing tenure", ["Renter", "Mortgagors", "Outright owners", "Other tenure"], None),
        ("labour_status", "Labour-market status", ["Employed", "Unemployed", "Inactive", "Retired"], None),
        ("borrowing_type", "Household borrowing", ["No borrowing", "Consumer credit", "Mortgage", "Both"], None),
        ("liquidity_status", "Consumption liquidity constraint", ["Liquidity constrained", "Not constrained"], panel["liquidity_status"].notna()),
        ("household_type", "Household type", ["Poor HtM", "Rich HtM", "Doubly wealthy", "Other"], None),
    ]

    BLUE, INK, GREY = "#2B5C8A", "#14213D", "#6A7179"
    n_columns = 5
    n_rows = int(np.ceil(len(plot_specs) / n_columns))
    fig, axes = plt.subplots(n_rows, n_columns, figsize=(18, 3.8 * n_rows), sharey=True, squeeze=False)
    for ax, (column, title, order, mask) in zip(axes.flat, plot_specs):
        sample = panel if mask is None else panel.loc[mask]
        distributions, labels = [], []
        for category in order:
            values = sample.loc[sample[column].astype("string") == str(category), "real_mpc_impact"].dropna()
            if not values.empty:
                distributions.append(values)
                labels.append(category)
        positions = np.arange(1, len(distributions) + 1)
        boxes = ax.boxplot(distributions, positions=positions, widths=0.58, showfliers=False, patch_artist=True)
        for patch in boxes["boxes"]:
            patch.set(facecolor=BLUE, alpha=0.75, edgecolor=BLUE)
        for median in boxes["medians"]:
            median.set(color=INK, linewidth=1.8)
        for line in boxes["whiskers"] + boxes["caps"]:
            line.set(color=BLUE, linewidth=1.0)
        ax.scatter(positions, [values.mean() for values in distributions], color="#111111", marker="o", s=20, zorder=4)
        ax.axhline(0, color=GREY, linewidth=0.7)
        display_labels = [
            {"No borrowing": "No\nborrowing", "Consumer credit": "Consumer\ncredit"}.get(label, label)
            for label in labels
        ]
        ax.set_xticks(positions, display_labels, rotation=0, ha="center", fontsize=8)
        ax.set_title(title, fontsize=10)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", color="#D9DEE5", linewidth=0.55, alpha=0.8)
        ax.set_axisbelow(True)
    for ax in axes.flat[len(plot_specs):]:
        ax.set_visible(False)
    fig.supylabel("MPC", x=0.008)

    from matplotlib.lines import Line2D
    fig.legend(handles=[Line2D([0], [0], marker="o", linestyle="", color="#111111", label="Mean")],
               loc="upper center", bbox_to_anchor=(0.5, 0.955), frameon=False)
    fig.suptitle("Impact MPC after a temporary income transfer", fontsize=14, y=0.995)
    fig.tight_layout(rect=(0.035, 0.02, 1, 0.91), h_pad=1.2, w_pad=1.0)
    plots_dir = outputs["analysis_dir"] / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(plots_dir / "paper_real_impact_mpc_income_panels.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
tolerance = 0.10  # try 0.01, 0.05 or 0.10
identified_cumulative = panel["labor_income_gap_share"] <= tolerance

## Impact MPC by payment group: income, all-debt NLA and liquid assets

The three panels rank households by **pre-transfer income**, **all-debt net liquid assets** and **liquid financial assets**. Here, all-debt NLA = liquid financial assets − consumer debt − mortgage debt. The existing consumer-debt-only NLA variable used by older plots is preserved. Liquid assets are the model's `liquid_financial_assets` balance, without subtracting debt. Each ranking divides the same filtered, pooled household-seed sample into fifths; equal values may be split between quintiles by the existing row-order tie rule.

Coloured bars show **mean real goods impact MPC within each group**, not the group's contribution to the overall mean. Black diamonds show the overall quintile mean. Group shares appear in the table below: a group with a high MPC may have few members. Very small groups (fewer than 30 household-seed observations) are omitted from the bars, but their counts/means remain in the exported CSV. Counts include repeated household observations across seeds; they are not independent-person sample sizes or confidence intervals.

Groups are assigned using the **baseline run without the transfer, in the quarter when the transfer is introduced**. We inspect whether the model paid support, raised spending to its minimum, cut spending or left debt payments unpaid. This is what “classification by payment outcomes” means. We do not classify households from their observed MPC or change groups using the transfer run.

| Group | What it means |
|---|---|
| **minimum-income top-up** | In the current minimum-income configuration, the household receives a positive minimum-income transfer in the baseline shock quarter. This replaces legacy Stage 5 basic-needs support. |

| **minimum consumption, self-funded** | The household receives no minimum-income top-up. Planned goods spending is at or below the minimum, and the model raises or keeps spending at that minimum using the household's own resources. |
| **constrained** | Neither group above applies. The model reduces planned spending during the payment step, or the household leaves a consumer-loan or mortgage payment unpaid. Low asset holdings alone do not determine this label. |
| **unconstrained** | None of those three conditions occurs. The model records no spending reduction or unpaid debt payment in this classification; this does not imply unlimited borrowing or no economic constraints. |

The minimum-income top-up category is checked first; among households without a top-up, the self-funded minimum-consumption category is checked before constrained and unconstrained outcomes. Monetary comparisons allow only the code's small numerical tolerance. The figure uses all households passing the ordinary sample filters, without the cumulative-MPC labour-income restriction. The outcome is the transfer-quarter difference in realised VAT-exclusive goods consumption, deflated using each run's CPI, per unit of real transfer.

The groups describe the **current model**. Whether the model should impose minimum consumption, and whether the desired-consumption equation needs adjustment for low-spending households, remain separate research questions; this plot changes neither mechanism.


In [ ]:
# Mean impact MPC by baseline payment group: three pre-shock rankings.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from src.mpc_analysis import assign_quantile_bin

analysis_dir = Path("data/output_data/mpc-experiment/analysis")
if globals().get("outputs") is not None:
    analysis_dir = Path(outputs["analysis_dir"])
group_panel = pd.read_csv(analysis_dir / "household_mpc_panel.csv")
asset_columns = ["liquid_financial_assets", "consumption_loan_debt", "mortgage_debt"]
if not set(asset_columns).issubset(group_panel.columns):
    # Older panels did not export the debt components. Use only the verified
    # same-run sidecar, not an older full HDF5 simulation.
    sidecar = pd.read_csv(analysis_dir / "household_pre_shock_liquidity_metadata.csv")
    group_panel = group_panel.merge(sidecar, on=["seed", "household_id"], how="left", validate="one_to_one")
    for saved, reference in [("income", "reference_income"), ("debt", "reference_debt")]:
        if not np.allclose(group_panel[saved], group_panel[reference], rtol=1e-12, atol=1e-6):
            raise ValueError("Liquidity metadata does not match this MPC run; regenerate the sidecar.")
if not np.isfinite(group_panel[asset_columns].to_numpy()).all():
    raise ValueError("Missing or non-finite pre-shock asset/debt values.")
if not np.allclose(
    group_panel["debt"], group_panel["consumption_loan_debt"] + group_panel["mortgage_debt"], rtol=1e-12, atol=1e-6
):
    raise ValueError("Debt components do not reconcile.")
if not np.allclose(
    group_panel["net_liquid_financial_assets"],
    group_panel["liquid_financial_assets"] - group_panel["consumption_loan_debt"],
    rtol=1e-12,
    atol=1e-6,
):
    raise ValueError("Liquid assets do not reconcile with the saved legacy NLA measure.")
group_panel["nla_all_debt"] = group_panel["liquid_financial_assets"] - group_panel["debt"]
rankings = [
    ("income", "Income quintile"),
    ("nla_all_debt", "All-debt NLA quintile"),
    ("liquid_financial_assets", "Liquid financial assets quintile"),
]
groups = [
    ("floor_bound_self_funded", "minimum consumption, self-funded", "#9369A8"),
    ("liquidity_constrained", "constrained", "#287C8E"),
    ("unconstrained", "unconstrained", "#74914B"),
]
# Minimum-income top-ups replace the legacy Stage 5 basic-needs support category.
minimum_income_group = ("minimum_income_recipient", "minimum income top-up", "#B24A5B")
if group_panel["mpc_regime"].eq(minimum_income_group[0]).any():
    groups.insert(0, minimum_income_group)
if not group_panel["mpc_regime"].isin([g[0] for g in groups]).all():
    raise ValueError("Unknown payment groups must be resolved before plotting.")
quintiles = ["Q1", "Q2", "Q3", "Q4", "Q5"]
x = np.arange(5)
bar_slot = 0.76 / len(groups)
fig, axes = plt.subplots(1, 3, figsize=(19, 6), sharey=True)
records = []
for ax, (variable, title) in zip(axes, rankings):
    bins = assign_quantile_bin(group_panel[variable])
    for i, (regime, label, color) in enumerate(groups):
        values = []
        for q in quintiles:
            qmask = bins.eq(q)
            members = group_panel.loc[qmask & group_panel["mpc_regime"].eq(regime), "real_mpc_impact"]
            n = len(members)
            # A mean based on fewer than 30 household-seed observations is
            # shown in the table, but not plotted as a stable group mean.
            values.append(members.mean() if n >= 30 else np.nan)
            records.append(
                dict(
                    ranking=title,
                    quintile=q,
                    group=label,
                    n=n,
                    share=100 * n / int(qmask.sum()),
                    mean_impact=members.mean(),
                )
            )
        ax.bar(x + (i - (len(groups) - 1) / 2) * bar_slot, values, width=bar_slot - 0.01, color=color, label=label)
    overall = [group_panel.loc[bins.eq(q), "real_mpc_impact"].mean() for q in quintiles]
    ax.scatter(x, overall, color="#171717", marker="D", s=38, zorder=5, label="all households")
    ax.set(title=title, xticks=x, xticklabels=quintiles, xlabel="Pre-shock quintile (Q1 = lowest)")
    ax.axhline(0, color="#777777", linewidth=0.7)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Mean real goods impact MPC")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.005))
fig.suptitle("Impact MPC by baseline payment group", fontsize=16)
fig.tight_layout(rect=(0, 0.16, 1, 0.94))
plots_dir = analysis_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(plots_dir / "mpc_impact_payment_groups_three_rankings.png", dpi=150, bbox_inches="tight")
plt.show()
group_breakdown = pd.DataFrame(records)
group_breakdown.to_csv(analysis_dir / "household_mpc_group_breakdown_three_rankings.csv", index=False)
print(
    "Group shares (%); see the CSV for counts and mean impact MPCs. Bars with fewer than 30 household-seed observations are omitted."
)
display(group_breakdown.pivot(index=["ranking", "quintile"], columns="group", values="share").round(1))

### Paper diagnostics
New runs write `analysis/paper_mpc_summary_raw.csv` and `paper_mpc_summary_filtered.csv`. Aggregate MPC uses transfer-size weights: sum of consumption changes divided by sum of transfers. `seed_cluster_se` clusters uncertainty by simulation seed. Household means and negative shares remain unweighted.
`paper_mpc_cashflows_*.csv` contains paired impact-window cash-flow snapshots when available; missing fields are omitted, never replaced by zeros. New cash-flow data require rerunning the experiment.

In [ ]:
# Rebuild paper outputs from saved panels without rerunning the simulation.
from pathlib import Path
from src.mpc_paper import write_paper_summaries, write_negative_mpc_diagnostics
from src.mpc_paper_figures import write_paper_figures
paper_dir = Path(outputs["analysis_dir"]) if outputs else Path(OUTPUT_DIR) / "analysis"
for sample in ("raw", "filtered"):
    paper_panel = pd.read_csv(paper_dir / f"household_mpc_panel_{sample}.csv")
    write_paper_summaries(paper_panel, paper_dir, sample=sample)
    write_negative_mpc_diagnostics(paper_panel, paper_dir, sample=sample)
    if sample == "filtered":
        write_paper_figures(paper_panel, paper_dir / "plots")
print(paper_dir)